# SETU-RAG — Speech-to-Speech (VANI) walkthrough
Audio in/out across 22 Indian languages + English, code-switched. Run on a **T4**.

## 1 · Install (speech deps)

In [ ]:
!pip -q install -r requirements.txt
# IndicConformer (NeMo) + Parler-TTS from source:
# !pip -q install git+https://github.com/huggingface/parler-tts.git
# IndicConformer: see github.com/AI4Bharat/IndicConformerASR

## 2 · Speech-side metrics (no GPU needed)

In [ ]:
from setu_rag.eval.speech_metrics import wer, cer, tts_intelligibility
print('WER', wer('mera refund kab aayega','mera refund kab ayega'))
print('CER', cer('refund','refnd'))

## 3 · CMI-conditioned TTS style description  [speech novelty #2]

In [ ]:
from setu_rag.speech.tts import style_description
print(style_description('hin_Deva', 0.37))

## 4 · Acoustic + lexical LID fusion  [speech novelty #1]

In [ ]:
from setu_rag.front_end.language_id import LanguageIdentifier
from setu_rag.speech.lid_fusion import fuse
lid = LanguageIdentifier().load()
toks = lid.tag('mera refund kab tak aayega order cancel kiya')
print(fuse({'hin':0.8,'eng':0.2}, toks, alpha=0.4))

## 5 · Full speech turn (runnable)
`build_pipeline()` makes the text core; `SpeechSetuRAG` wraps it with ASR (IndicConformer→Whisper)
in front and CMI-conditioned TTS behind. ASR/TTS load on demand and are freed between stages to fit a T4.

In [ ]:
import sys, os; sys.path.insert(0, os.getcwd())
from setu_rag.app import build_pipeline
from setu_rag.speech_pipeline import SpeechSetuRAG

rag = build_pipeline()                 # text core (real models on a GPU)
voice = SpeechSetuRAG(rag)

# Provide a 16 kHz mono clip of a code-switched question:
turn = voice.answer_file('question.wav', '/content/reply.wav')
print(turn.transcript, '->', turn.answer_text)
print('matrix:', turn.matrix_lang, '| cmi:', round(turn.cmi, 2), '| timings(ms):', turn.timings_ms)
# reply audio is written to /content/reply.wav (if Parler-TTS is installed)